In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split

# --- Load the datasets ---

# Define the base path for the data files
# In a real Kaggle environment, this would typically be '../input/house-prices-advanced-regression-techniques/'
# We are using the path you provided.
base_path = '/Kaggle/input/house prices advanced regression techniques/'

# Construct the full paths to the training and test CSV files
train_csv_path = os.path.join(base_path, '/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test_csv_path = os.path.join(base_path, '/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

# Use a try-except block to handle potential FileNotFoundError
try:
    # Load train.csv into a DataFrame named train_df
    train_df = pd.read_csv(train_csv_path)

    # Load test.csv into a DataFrame named test_df
    test_df = pd.read_csv(test_csv_path)

    # --- Display confirmation and initial view of the data ---
    print("Files loaded successfully!")
    print("-" * 30)

    # Print the shape (rows, columns) of each DataFrame
    print(f"Training data shape: {train_df.shape}")
    print(f"Test data shape: {test_df.shape}")
    print("-" * 30)

    # Display the first 5 rows of the training data to verify it's loaded correctly
    print("First 5 rows of the training data (train_df):")
    print(train_df.head())
    print("\n" + "-" * 30)

    # Display the first 5 rows of the test data
    print("First 5 rows of the test data (test_df):")
    print(test_df.head())

    # --- Exploratory Data Analysis (EDA) ---
    print("\n\n" + "="*50)
    print("Exploratory Data Analysis (EDA) on train_df")
    print("="*50)

    # 1. Basic Structure and Statistics
    print("\n--- 1. Data Structure (train_df.info()) ---")
    # This gives us a summary of the DataFrame, including data types and non-null values.
    train_df.info()

    print("\n--- 2. Descriptive Statistics (train_df.describe()) ---")
    # This provides summary statistics for all numerical columns.
    print(train_df.describe())

    # 2. Visualize SalePrice Distribution
    print("\n--- Plotting SalePrice Distribution ---")
    # Set the aesthetic style of the plots
    sns.set_style('whitegrid')
    plt.figure(figsize=(16, 6))

    # Plot raw SalePrice distribution
    plt.subplot(1, 2, 1)
    sns.histplot(train_df['SalePrice'], kde=True, bins=50, color='blue')
    plt.title('Distribution of SalePrice (Raw)', fontsize=15)
    plt.xlabel('Sale Price ($)', fontsize=12)
    plt.ylabel('Frequency', fontsize=12)

    # The SalePrice is skewed. Log-transforming it can help normalize the distribution,
    # which often improves the performance of regression models.
    # We use np.log1p which calculates log(1+x) to handle potential zero values.
    train_df['SalePrice_log'] = np.log1p(train_df['SalePrice'])

    # Plot log-transformed SalePrice distribution
    plt.subplot(1, 2, 2)
    sns.histplot(train_df['SalePrice_log'], kde=True, bins=50, color='green')
    plt.title('Distribution of SalePrice (Log-Transformed)', fontsize=15)
    plt.xlabel('Log of Sale Price', fontsize=12)
    plt.ylabel('Frequency', fontsize=12)

    plt.tight_layout()
    plt.show()

    # 3. Correlation Heatmap
    print("\n--- Plotting Correlation Heatmap ---")
    # Identify numerical columns to calculate correlation
    numeric_features = train_df.select_dtypes(include=np.number)
    correlation_matrix = numeric_features.corr()

    # Find the top 10 features most correlated with SalePrice
    # We select 11 because 'SalePrice' itself will be included.
    top_corr_cols = correlation_matrix.nlargest(11, 'SalePrice')['SalePrice'].index
    top_corr_matrix = train_df[top_corr_cols].corr()

    plt.figure(figsize=(12, 10))
    sns.heatmap(top_corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
    plt.title('Correlation Heatmap of Top Features', fontsize=16)
    plt.show()

    # 4. Boxplots for Categorical Variables
    print("\n--- Plotting Categorical Feature Boxplots ---")
    # Boxplots are great for visualizing the distribution of a numerical variable
    # across different categories of a categorical variable.
    plt.figure(figsize=(18, 8))
    sns.boxplot(x='Neighborhood', y='SalePrice', data=train_df)
    plt.xticks(rotation=45, ha='right')
    plt.title('Sale Price Distribution by Neighborhood', fontsize=16)
    plt.xlabel('Neighborhood', fontsize=12)
    plt.ylabel('Sale Price ($)', fontsize=12)
    plt.tight_layout()
    plt.show()

    # --- Feature Selection and Data Splitting ---
    print("\n\n" + "="*50)
    print("Preparing Data for Modeling")
    print("="*50)

    # 1. Define Target and Features
    # The target variable is 'SalePrice'.
    y = train_df['SalePrice']

    # For features (X), we'll start with only the numeric columns for simplicity.
    # We drop the target variables and the ID column.
    X = train_df.select_dtypes(include=np.number)
    X = X.drop(columns=['Id', 'SalePrice', 'SalePrice_log'])

    # 2. Handle Missing Values in Features
    # A simple strategy is to fill missing numerical values with the median of their column.
    print(f"\nMissing values in X before imputation: {X.isnull().sum().sum()}")
    for col in X.columns:
        if X[col].isnull().any():
            median_val = X[col].median()
            X[col] = X[col].fillna(median_val)
    print(f"Missing values in X after imputation: {X.isnull().sum().sum()}")

    # 3. Split the Data
    # We split the data into training (80%) and validation (20%) sets.
    # random_state ensures that the split is the same every time we run the code.
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    print("\n--- Data Splitting Complete ---")
    print(f"Shape of X_train: {X_train.shape}")
    print(f"Shape of X_test: {X_test.shape}")
    print(f"Shape of y_train: {y_train.shape}")
    print(f"Shape of y_test: {y_test.shape}")

    # --- Model Training: Linear Regression ---
    print("\n\n" + "="*50)
    print("Training a Linear Regression Model")
    print("="*50)

    # 1. Import the model and metrics
    from sklearn.linear_model import LinearRegression
    from sklearn.metrics import mean_squared_error, r2_score

    # 2. Create an instance of the model
    lr_model = LinearRegression()

    # 3. Train the model on the training data
    print("\nFitting the model...")
    lr_model.fit(X_train, y_train)
    print("Model training complete.")

    # 4. Make predictions on the validation set
    print("Making predictions on the test set...")
    y_pred = lr_model.predict(X_test)

    # 5. Evaluate the model's performance
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    print("\n--- Model Evaluation ---")
    print(f"R^2 Score: {r2:.4f}")
    print(f"Mean Squared Error (MSE): {mse:.2f}")
    print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")

    # 6. Visualize Predictions vs. Actual Values
    print("\n--- Plotting Predictions vs. Actual Values ---")
    plt.figure(figsize=(10, 8))
    plt.scatter(y_test, y_pred, alpha=0.5)
    plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color='red', linewidth=2) # Diagonal line
    plt.title('Actual vs. Predicted Sale Prices (Linear Regression)', fontsize=16)
    plt.xlabel('Actual Sale Price ($)', fontsize=12)
    plt.ylabel('Predicted Sale Price ($)', fontsize=12)
    plt.grid(True)
    plt.show()


except FileNotFoundError as e:
    print(f"Error: {e}")
    print("Please make sure the file paths are correct and the files exist.")
    print(f"Attempted to load train.csv from: {train_csv_path}")
    print(f"Attempted to load test.csv from: {test_csv_path}")



In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import xgboost as xgb

# --- Load the datasets ---

# Define the base path for the data files
# In a real Kaggle environment, this would typically be '../input/house-prices-advanced-regression-techniques/'
# We are using the path you provided.
base_path = '/Kaggle/input/house prices advanced regression techniques/'

# Construct the full paths to the training and test CSV files
train_csv_path = os.path.join(base_path, '/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test_csv_path = os.path.join(base_path, '/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

# Use a try-except block to handle potential FileNotFoundError
try:
    # Load train.csv into a DataFrame named train_df
    train_df = pd.read_csv(train_csv_path)

    # Load test.csv into a DataFrame named test_df
    test_df = pd.read_csv(test_csv_path)

    # --- Display confirmation and initial view of the data ---
    print("Files loaded successfully!")
    print("-" * 30)

    # Print the shape (rows, columns) of each DataFrame
    print(f"Training data shape: {train_df.shape}")
    print(f"Test data shape: {test_df.shape}")
    print("-" * 30)

    # Display the first 5 rows of the training data to verify it's loaded correctly
    print("First 5 rows of the training data (train_df):")
    print(train_df.head())
    print("\n" + "-" * 30)

    # Display the first 5 rows of the test data
    print("First 5 rows of the test data (test_df):")
    print(test_df.head())

    # --- Exploratory Data Analysis (EDA) ---
    print("\n\n" + "="*50)
    print("Exploratory Data Analysis (EDA) on train_df")
    print("="*50)

    # 1. Basic Structure and Statistics
    print("\n--- 1. Data Structure (train_df.info()) ---")
    # This gives us a summary of the DataFrame, including data types and non-null values.
    train_df.info()

    print("\n--- 2. Descriptive Statistics (train_df.describe()) ---")
    # This provides summary statistics for all numerical columns.
    print(train_df.describe())

    # 2. Visualize SalePrice Distribution
    print("\n--- Plotting SalePrice Distribution ---")
    # Set the aesthetic style of the plots
    sns.set_style('whitegrid')
    plt.figure(figsize=(16, 6))

    # Plot raw SalePrice distribution
    plt.subplot(1, 2, 1)
    sns.histplot(train_df['SalePrice'], kde=True, bins=50, color='blue')
    plt.title('Distribution of SalePrice (Raw)', fontsize=15)
    plt.xlabel('Sale Price ($)', fontsize=12)
    plt.ylabel('Frequency', fontsize=12)

    # The SalePrice is skewed. Log-transforming it can help normalize the distribution,
    # which often improves the performance of regression models.
    # We use np.log1p which calculates log(1+x) to handle potential zero values.
    train_df['SalePrice_log'] = np.log1p(train_df['SalePrice'])

    # Plot log-transformed SalePrice distribution
    plt.subplot(1, 2, 2)
    sns.histplot(train_df['SalePrice_log'], kde=True, bins=50, color='green')
    plt.title('Distribution of SalePrice (Log-Transformed)', fontsize=15)
    plt.xlabel('Log of Sale Price', fontsize=12)
    plt.ylabel('Frequency', fontsize=12)

    plt.tight_layout()
    plt.show()

    # 3. Correlation Heatmap
    print("\n--- Plotting Correlation Heatmap ---")
    # Identify numerical columns to calculate correlation
    numeric_features = train_df.select_dtypes(include=np.number)
    correlation_matrix = numeric_features.corr()

    # Find the top 10 features most correlated with SalePrice
    # We select 11 because 'SalePrice' itself will be included.
    top_corr_cols = correlation_matrix.nlargest(11, 'SalePrice')['SalePrice'].index
    top_corr_matrix = train_df[top_corr_cols].corr()

    plt.figure(figsize=(12, 10))
    sns.heatmap(top_corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
    plt.title('Correlation Heatmap of Top Features', fontsize=16)
    plt.show()

    # 4. Boxplots for Categorical Variables
    print("\n--- Plotting Categorical Feature Boxplots ---")
    # Boxplots are great for visualizing the distribution of a numerical variable
    # across different categories of a categorical variable.
    plt.figure(figsize=(18, 8))
    sns.boxplot(x='Neighborhood', y='SalePrice', data=train_df)
    plt.xticks(rotation=45, ha='right')
    plt.title('Sale Price Distribution by Neighborhood', fontsize=16)
    plt.xlabel('Neighborhood', fontsize=12)
    plt.ylabel('Sale Price ($)', fontsize=12)
    plt.tight_layout()
    plt.show()

    # --- Feature Engineering ---
    print("\n\n" + "="*50)
    print("Creating New Features")
    print("="*50)
    # We will create the same features for both train and test sets to maintain consistency.
    for df in [train_df, test_df]:
        df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
        df['TotalBath'] = df['FullBath'] + (0.5 * df['HalfBath']) + df['BsmtFullBath'] + (0.5 * df['BsmtHalfBath'])
        df['HouseAge'] = df['YrSold'] - df['YearBuilt']
        df['YearsSinceRemod'] = df['YrSold'] - df['YearRemodAdd']

    print("New features created: TotalSF, TotalBath, HouseAge, YearsSinceRemod")
    print(train_df[['TotalSF', 'TotalBath', 'HouseAge', 'YearsSinceRemod']].head())


    # --- Feature Selection and Data Splitting ---
    print("\n\n" + "="*50)
    print("Preparing Data for Modeling")
    print("="*50)

    # 1. Define Target and Features
    # The target variable is 'SalePrice'.
    y = train_df['SalePrice']

    # For features (X), we use the updated numeric columns, including our new ones.
    # We drop the target variables and the ID column.
    X = train_df.select_dtypes(include=np.number)
    X = X.drop(columns=['Id', 'SalePrice', 'SalePrice_log'])

    # 2. Handle Missing Values in Features
    # A simple strategy is to fill missing numerical values with the median of their column.
    print(f"\nMissing values in X before imputation: {X.isnull().sum().sum()}")
    for col in X.columns:
        if X[col].isnull().any():
            median_val = X[col].median()
            X[col] = X[col].fillna(median_val)
    print(f"Missing values in X after imputation: {X.isnull().sum().sum()}")

    # 3. Split the Data
    # We split the data into training (80%) and validation (20%) sets.
    # random_state ensures that the split is the same every time we run the code.
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    print("\n--- Data Splitting Complete ---")
    print(f"Shape of X_train: {X_train.shape}")
    print(f"Shape of X_test: {X_test.shape}")
    print(f"Shape of y_train: {y_train.shape}")
    print(f"Shape of y_test: {y_test.shape}")


    # --- Model Training and Evaluation ---
    print("\n\n" + "="*50)
    print("1. Linear Regression with New Features")
    print("="*50)

    # 1. Initialize and Train the Model
    # Create an instance of the LinearRegression model
    linear_model = LinearRegression()

    # Train the model on the training data
    print("Training the Linear Regression model...")
    linear_model.fit(X_train, y_train)
    print("Training complete.")

    # 2. Make Predictions
    # Use the trained model to make predictions on the validation set (X_test)
    y_pred = linear_model.predict(X_test)

    # 3. Evaluate the Model
    # Calculate R^2 Score, Mean Squared Error, and Root Mean Squared Error (RMSE)
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)

    print("\n--- Model Performance with New Features ---")
    print(f"R-squared (R²): {r2:.4f}")
    print(f"Mean Squared Error (MSE): ${mse:,.2f}")
    print(f"Root Mean Squared Error (RMSE): ${rmse:,.2f}")

    # --- 2. Random Forest Regressor ---
    print("\n\n" + "="*50)
    print("2. Random Forest Regressor")
    print("="*50)

    # Initialize and Train
    print("Training the Random Forest model...")
    # n_jobs=-1 uses all available CPU cores to speed up training
    rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train, y_train)
    print("Training complete.")

    # Predict and Evaluate
    y_pred_rf = rf_model.predict(X_test)
    rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))

    print("\n--- Model Performance ---")
    print(f"Random Forest RMSE: ${rmse_rf:,.2f}")

    # --- 3. Gradient Boosting Regressor ---
    print("\n\n" + "="*50)
    print("3. Gradient Boosting Regressor")
    print("="*50)

    # Initialize and Train
    print("Training the Gradient Boosting model...")
    gb_model = GradientBoostingRegressor(n_estimators=100, random_state=42)
    gb_model.fit(X_train, y_train)
    print("Training complete.")

    # Predict and Evaluate
    y_pred_gb = gb_model.predict(X_test)
    rmse_gb = np.sqrt(mean_squared_error(y_test, y_pred_gb))

    print("\n--- Model Performance ---")
    print(f"Gradient Boosting RMSE: ${rmse_gb:,.2f}")

    # --- 4. XGBoost Regressor ---
    print("\n\n" + "="*50)
    print("4. XGBoost Regressor")
    print("="*50)

    # Initialize and Train
    print("Training the XGBoost model...")
    # XGBoost is a powerful and popular library for gradient boosting
    xgb_model = xgb.XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    xgb_model.fit(X_train, y_train)
    print("Training complete.")

    # Predict and Evaluate
    y_pred_xgb = xgb_model.predict(X_test)
    rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))

    print("\n--- Model Performance ---")
    print(f"XGBoost RMSE: ${rmse_xgb:,.2f}")


except FileNotFoundError as e:
    print(f"Error: {e}")
    print("Please make sure the file paths are correct and the files exist.")
    print(f"Attempted to load train.csv from: {train_csv_path}")
    print(f"Attempted to load test.csv from: {test_csv_path}")



In [ ]:
# -*- coding: utf-8 -*-
"""
This script provides a comprehensive solution for the Kaggle "House Prices:
Advanced Regression Techniques" competition. It covers the entire machine
learning pipeline, including data loading, exploratory data analysis (EDA),
feature engineering, model training, and evaluation.

Key Steps:
1.  **Load Data**: Reads the training and test datasets.
2.  **Exploratory Data Analysis (EDA)**:
    -   Analyzes the target variable ('SalePrice') distribution and applies a
        log transformation to normalize it.
    -   Visualizes correlations between features and the target variable.
    -   Examines relationships between categorical features and SalePrice.
3.  **Feature Engineering & Preprocessing**:
    -   Creates new features (e.g., Total Square Feet, House Age).
    -   Handles missing values (imputation) for both numerical and categorical
        features.
    -   Applies one-hot encoding to categorical variables to convert them into
        a machine-readable format.
4.  **Model Training**:
    -   Splits the data into training and validation sets.
    -   Trains and evaluates several regression models:
        -   Linear Regression
        -   Random Forest Regressor
        -   Gradient Boosting Regressor
        -   XGBoost Regressor
5.  **Evaluation**:
    -   Measures model performance using Root Mean Squared Error (RMSE).

This structured approach ensures that the data is well understood and prepared,
leading to more robust and accurate model predictions.
"""

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
import xgboost as xgb

# --- Configuration ---
# In a real Kaggle environment, this path is typically '/kaggle/input/house-prices-advanced-regression-techniques/'
# Using a relative path is often more portable.
BASE_PATH = './'
TRAIN_CSV_PATH = os.path.join(BASE_PATH, '/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
TEST_CSV_PATH = os.path.join(BASE_PATH, '/kaggle/input/house-prices-advanced-regression-techniques/test.csv')


def load_data(train_path, test_path):
    """
    Loads the training and test datasets from specified CSV files.

    Args:
        train_path (str): The file path for the training data.
        test_path (str): The file path for the test data.

    Returns:
        tuple: A tuple containing two pandas DataFrames (train_df, test_df).
               Returns (None, None) if files are not found.
    """
    try:
        train_df = pd.read_csv(train_path)
        test_df = pd.read_csv(test_path)
        print("Files loaded successfully!")
        print("-" * 30)
        print(f"Training data shape: {train_df.shape}")
        print(f"Test data shape: {test_df.shape}")
        return train_df, test_df
    except FileNotFoundError as e:
        print(f"Error loading data: {e}")
        print("Please ensure the CSV files are in the correct directory.")
        return None, None


def run_exploratory_data_analysis(df):
    """
    Performs and visualizes key aspects of the training data.

    Args:
        df (pd.DataFrame): The training DataFrame.
    """
    print("\n" + "="*50)
    print("Exploratory Data Analysis (EDA)")
    print("="*50)

    # --- 1. Analyze SalePrice Distribution ---
    print("\n--- Plotting SalePrice Distribution ---")
    plt.figure(figsize=(16, 6))
    sns.set_style('whitegrid')

    # Plot raw SalePrice
    plt.subplot(1, 2, 1)
    sns.histplot(df['SalePrice'], kde=True, bins=50, color='blue')
    plt.title('Distribution of SalePrice (Raw)', fontsize=15)
    plt.xlabel('Sale Price ($)', fontsize=12)
    plt.ylabel('Frequency', fontsize=12)

    # Plot log-transformed SalePrice
    # Log transform helps normalize the distribution, which benefits many models.
    plt.subplot(1, 2, 2)
    sns.histplot(np.log1p(df['SalePrice']), kde=True, bins=50, color='green')
    plt.title('Distribution of SalePrice (Log-Transformed)', fontsize=15)
    plt.xlabel('Log of Sale Price', fontsize=12)
    plt.ylabel('Frequency', fontsize=12)
    plt.tight_layout()
    plt.show()

    # --- 2. Correlation Heatmap ---
    print("\n--- Plotting Correlation Heatmap ---")
    numeric_features = df.select_dtypes(include=np.number)
    correlation_matrix = numeric_features.corr()

    # Get the top 11 features most correlated with SalePrice
    top_corr_cols = correlation_matrix.nlargest(11, 'SalePrice')['SalePrice'].index
    top_corr_matrix = df[top_corr_cols].corr()

    plt.figure(figsize=(12, 10))
    sns.heatmap(top_corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
    plt.title('Correlation Heatmap of Top Features vs. SalePrice', fontsize=16)
    plt.show()


def engineer_features(df):
    """
    Creates new, potentially more informative features from existing data.

    Args:
        df (pd.DataFrame): The DataFrame to add features to.

    Returns:
        pd.DataFrame: The DataFrame with new features.
    """
    print("\n" + "="*50)
    print("Creating New Features")
    print("="*50)

    # Combine square footage features
    df['TotalSF'] = df.get('TotalBsmtSF', 0) + df.get('1stFlrSF', 0) + df.get('2ndFlrSF', 0)

    # Combine bathroom features
    df['TotalBath'] = (df.get('FullBath', 0) +
                       0.5 * df.get('HalfBath', 0) +
                       df.get('BsmtFullBath', 0) +
                       0.5 * df.get('BsmtHalfBath', 0))

    # Calculate age and time since remodel
    df['HouseAge'] = df['YrSold'] - df['YearBuilt']
    df['YearsSinceRemod'] = df['YrSold'] - df['YearRemodAdd']

    print("New features created: TotalSF, TotalBath, HouseAge, YearsSinceRemod")
    return df


def preprocess_data(train_df, test_df):
    """
    Handles missing values and encodes categorical features for modeling.

    Args:
        train_df (pd.DataFrame): The training DataFrame.
        test_df (pd.DataFrame): The test DataFrame.

    Returns:
        tuple: A tuple of processed DataFrames (X, X_test_processed).
    """
    print("\n" + "="*50)
    print("Preprocessing Data")
    print("="*50)

    # --- 1. Align columns and define target/features ---
    y = np.log1p(train_df['SalePrice'])
    # Drop target and ID from the training features
    X = train_df.drop(columns=['Id', 'SalePrice'])
    X_test_processed = test_df.drop(columns=['Id'])

    # Ensure both dataframes have the same columns, adding missing ones if necessary
    X, X_test_processed = X.align(X_test_processed, join='outer', axis=1, fill_value=0)

    # --- 2. Impute Missing Values ---
    # Numerical columns: fill with the median
    num_cols = X.select_dtypes(include=np.number).columns
    for col in num_cols:
        median_val = X[col].median()
        X[col].fillna(median_val, inplace=True)
        X_test_processed[col].fillna(median_val, inplace=True)

    # Categorical columns: fill with the mode (most frequent value)
    cat_cols = X.select_dtypes(include='object').columns
    for col in cat_cols:
        mode_val = X[col].mode()[0]
        X[col].fillna(mode_val, inplace=True)
        X_test_processed[col].fillna(mode_val, inplace=True)
    
    print("Missing values handled.")

    # --- 3. One-Hot Encode Categorical Features ---
    X = pd.get_dummies(X, columns=cat_cols, dummy_na=False)
    X_test_processed = pd.get_dummies(X_test_processed, columns=cat_cols, dummy_na=False)

    # Align columns again after one-hot encoding to ensure they match perfectly
    X, X_test_processed = X.align(X_test_processed, join='inner', axis=1)

    print("Categorical features one-hot encoded.")

    return X, y, X_test_processed


def train_and_evaluate_models(X, y):
    """
    Splits data, trains multiple models, and prints their performance.

    Args:
        X (pd.DataFrame): The feature matrix.
        y (pd.Series): The target vector.
    
    Returns:
        str: The name of the best performing model.
    """
    # Split the data into training and validation sets
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

    print("\n" + "="*50)
    print("Model Training & Evaluation")
    print("="*50)
    print(f"Training data shape: {X_train.shape}")
    print(f"Validation data shape: {X_val.shape}")

    models = {
        "Linear Regression": LinearRegression(),
        "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
        "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42),
        "XGBoost": xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1)
    }

    results = {}

    for name, model in models.items():
        print(f"\n--- Training {name} ---")
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)

        # We are predicting the log of SalePrice, so we calculate RMSE on the log values
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        results[name] = rmse
        print(f"{name} Log-RMSE: {rmse:.4f}")

    # --- Find and return the best model ---
    best_model_name = min(results, key=results.get)

    # --- Display Final Results ---
    print("\n" + "="*50)
    print("Model Comparison (Log-RMSE)")
    print("="*50)
    for name, score in sorted(results.items(), key=lambda item: item[1]):
        print(f"{name:<20} | {score:.4f}")
    
    print(f"\nThe best model is: {best_model_name}")
    return best_model_name


def generate_submission(best_model_name, X, y, X_test_processed, test_df):
    """
    Retrains the best model on the entire dataset, predicts on the test set,
    and generates a submission file.

    Args:
        best_model_name (str): The name of the best model to use.
        X (pd.DataFrame): The full training feature matrix.
        y (pd.Series): The full training target vector.
        X_test_processed (pd.DataFrame): The processed test feature matrix.
        test_df (pd.DataFrame): The original test DataFrame (for IDs).
    """
    print("\n" + "="*50)
    print("Generating Submission File")
    print("="*50)

    # --- 1. Re-initialize the best model ---
    # Using a dictionary to map model names to their constructors
    model_constructors = {
        "Linear Regression": LinearRegression(),
        "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
        "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42),
        "XGBoost": xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1)
    }
    best_model = model_constructors[best_model_name]
    
    # --- 2. Retrain the model on the entire training dataset ---
    print(f"Retraining the best model ({best_model_name}) on the entire dataset...")
    best_model.fit(X, y)
    print("Retraining complete.")

    # --- 3. Predict on the test data ---
    print("Predicting on the test set...")
    log_predictions = best_model.predict(X_test_processed)
    
    # --- 4. Inverse transform the predictions ---
    # np.expm1 is the inverse of np.log1p
    final_predictions = np.expm1(log_predictions)
    
    # --- 5. Create the submission DataFrame ---
    submission_df = pd.DataFrame({
        'Id': test_df['Id'],
        'SalePrice': final_predictions
    })

    # --- 6. Save the submission file ---
    submission_path = 'submission.csv'
    submission_df.to_csv(submission_path, index=False)
    print(f"\nSubmission file created successfully at: {submission_path}")
    print(submission_df.head())


def main():
    """
    Main function to execute the full machine learning pipeline.
    """
    # --- 1. Load Data ---
    train_df, test_df = load_data(TRAIN_CSV_PATH, TEST_CSV_PATH)
    if train_df is None or test_df is None:
        return  # Stop execution if data loading fails

    # --- 2. EDA ---
    run_exploratory_data_analysis(train_df.copy())

    # --- 3. Feature Engineering ---
    # Apply to both train and test sets for consistency
    train_df = engineer_features(train_df)
    test_df = engineer_features(test_df)

    # --- 4. Preprocessing ---
    X, y, X_test_processed = preprocess_data(train_df.copy(), test_df.copy())

    # --- 5. Model Training and Evaluation ---
    best_model_name = train_and_evaluate_models(X, y)

    # --- 6. Generate Submission File ---
    generate_submission(best_model_name, X, y, X_test_processed, test_df)


if __name__ == "__main__":
    main()

